# March Machine Learning Mania 2026 — Winning Predictor

**Strategy**: Elo (with home-court advantage) + Massey Ordinals (pre-tournament, no leakage)
+ Four-Factor Advanced Stats + XGBoost/LightGBM/LR Ensemble
+ Isotonic calibration fitted on OOF predictions

**Target Brier Score**: ≤ 0.21 (random baseline = 0.25, previous best = 0.24740)

In [ ]:
# ─── Cell 1: Imports & Configuration ────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
from collections import defaultdict
import re

from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

DATA_DIR = '/kaggle/input/march-machine-learning-mania-2026'

# ── Elo hyperparameters ──────────────────────────────────────────────────────
ELO_INITIAL    = 1500.0
ELO_K          = 20.0
ELO_CARRY_OVER = 0.75       # fraction of prior-season rating carried forward
ELO_HOME_ADV   = 100.0      # home-court advantage in Elo points
ELO_SCALE      = 400.0

# ── Massey rating systems with strong historical predictive power ────────────
TOP_MASSEY = ['POM', 'SAG', 'BPI', 'MOR', 'WLK', 'KPK', 'ESPN', 'RTH', 'DOK']

# ── Ensemble weights ─────────────────────────────────────────────────────────
WEIGHTS       = [0.4, 0.4, 0.2]   # XGB, LGB, LR
CLIP_LO, CLIP_HI = 0.05, 0.95

# ── Training season ranges ───────────────────────────────────────────────────
M_TRAIN_START = 2003
W_TRAIN_START = 2010

print('Setup complete.')

In [ ]:
# ─── Cell 2: Safe Data Loader ────────────────────────────────────────────────

def safe_read(fname, **kwargs):
    path = f'{DATA_DIR}/{fname}'
    if os.path.exists(path):
        df = pd.read_csv(path, **kwargs)
        print(f'  {fname:55s} → {df.shape}')
        return df
    print(f'  MISSING: {fname}')
    return pd.DataFrame()

print('Loading all data files...')
m_rs   = safe_read('MRegularSeasonResults.csv')
w_rs   = safe_read('WRegularSeasonResults.csv')
m_rsd  = safe_read('MRegularSeasonDetailedResults.csv')
w_rsd  = safe_read('WRegularSeasonDetailedResults.csv')
m_tour = safe_read('MNCAATourneyResults.csv')
w_tour = safe_read('WNCAATourneyResults.csv')
m_td   = safe_read('MNCAATourneyDetailedResults.csv')
w_td   = safe_read('WNCAATourneyDetailedResults.csv')
m_seeds  = safe_read('MNCAATourneySeeds.csv')
w_seeds  = safe_read('WNCAATourneySeeds.csv')
m_massey = safe_read('MMasseyOrdinals.csv')
w_massey = safe_read('WMasseyOrdinals.csv')
m_teams  = safe_read('MTeams.csv')
w_teams  = safe_read('WTeams.csv')
sample_sub = safe_read('SampleSubmission.csv')

print(f'\nSampleSubmission: {len(sample_sub)} rows')
print(sample_sub.head(3))

In [ ]:
# ─── Cell 3: Elo Rating Engine (with Home-Court Advantage) ───────────────────

def compute_elo(results_df):
    """
    Game-by-game Elo ratings with year-to-year carryover and home advantage.
    Returns dict: (Season, TeamID) -> end-of-season Elo
    """
    if results_df.empty:
        return {}

    elo       = {}          # {TeamID: current elo}
    season_elo = {}         # {(season, TeamID): end-of-season elo}
    has_wloc  = 'WLoc' in results_df.columns

    for season, grp in results_df.sort_values(['Season', 'DayNum']).groupby('Season'):
        # Year-to-year carryover: regress toward mean
        elo = {tid: ELO_INITIAL + ELO_CARRY_OVER * (r - ELO_INITIAL)
               for tid, r in elo.items()}

        for _, row in grp.iterrows():
            w = int(row['WTeamID'])
            l = int(row['LTeamID'])
            elo.setdefault(w, ELO_INITIAL)
            elo.setdefault(l, ELO_INITIAL)

            # Location-based advantage
            if has_wloc:
                loc = str(row.get('WLoc', 'N'))
                adv = ELO_HOME_ADV if loc == 'H' else (-ELO_HOME_ADV if loc == 'A' else 0)
            else:
                adv = 0

            E_w = 1.0 / (1.0 + 10 ** ((elo[l] - elo[w] - adv) / ELO_SCALE))
            elo[w] += ELO_K * (1.0 - E_w)
            elo[l] += ELO_K * (0.0 - (1.0 - E_w))

        # Snapshot end-of-season ratings
        for tid, r in elo.items():
            season_elo[(int(season), tid)] = r

    return season_elo


print('Computing Elo ratings...')
m_elo = compute_elo(m_rs)
w_elo = compute_elo(w_rs)
print(f'Men Elo keys: {len(m_elo)} | Women Elo keys: {len(w_elo)}')

# Sanity check: top teams in latest men's season
latest_m = max(k[0] for k in m_elo)
top10 = sorted([(k[1], v) for k, v in m_elo.items() if k[0] == latest_m],
               key=lambda x: -x[1])[:5]
print(f'Top 5 Elo (Men {latest_m}): {top10}')

In [ ]:
# ─── Cell 4: Regular Season Stats ────────────────────────────────────────────

def compute_season_stats(rs_df):
    """Win%, point diff, PPG, OppPPG, recent form from compact results."""
    if rs_df.empty:
        return pd.DataFrame()

    # Stack winner + loser rows
    w = rs_df[['Season','WTeamID','WScore','LScore']].copy()
    w.columns = ['Season','TeamID','Pts','OppPts']
    w['Win'] = 1

    l = rs_df[['Season','LTeamID','LScore','WScore']].copy()
    l.columns = ['Season','TeamID','Pts','OppPts']
    l['Win'] = 0

    all_g = pd.concat([w, l], ignore_index=True)
    all_g['Diff'] = all_g['Pts'] - all_g['OppPts']

    agg = all_g.groupby(['Season','TeamID']).agg(
        Games     = ('Win', 'count'),
        Wins      = ('Win',  'sum'),
        PointDiff = ('Diff', 'mean'),
        PPG       = ('Pts',  'mean'),
        OppPPG    = ('OppPts','mean'),
    ).reset_index()
    agg['WinPct'] = agg['Wins'] / agg['Games']

    # Recent form: last 14 games per team per season
    rs_sorted = rs_df.sort_values(['Season','DayNum'])
    recent = []
    for (season, tid), grp in all_g.groupby(['Season','TeamID']):
        last14 = all_g[
            (all_g['Season']==season) & (all_g['TeamID']==tid)
        ]['Win'].values[-14:]
        recent.append({'Season': season, 'TeamID': tid,
                       'RecentWinPct': float(np.mean(last14)) if len(last14) else 0.5})
    recent_df = pd.DataFrame(recent)

    agg = agg.merge(recent_df, on=['Season','TeamID'], how='left')
    return agg.set_index(['Season','TeamID'])


print('Computing regular season stats...')
m_stats = compute_season_stats(m_rs)
w_stats = compute_season_stats(w_rs)
print(f'Men stats: {m_stats.shape} | Women stats: {w_stats.shape}')

In [ ]:
# ─── Cell 5: Four-Factor Advanced Stats ──────────────────────────────────────

def compute_advanced_stats(rsd_df):
    """
    Four-factor efficiency stats from detailed box-score results:
      eFG%   = (FGM + 0.5*FGM3) / FGA
      TO%    = TO / possessions
      ORB%   = OR / (OR + opp_DR)
      FTRate = FTA / FGA
      AdjOE  = pts / poss * 100
      AdjDE  = opp_pts / poss * 100
    """
    if rsd_df is None or rsd_df.empty:
        return pd.DataFrame()

    records = []
    for _, r in rsd_df.iterrows():
        for side, opp in [('W','L'), ('L','W')]:
            tid     = int(r[f'{side}TeamID'])
            pts     = float(r.get(f'{side}Score', 0))
            opp_pts = float(r.get(f'{opp}Score', 0))
            fgm     = float(r.get(f'{side}FGM', 0))
            fga     = float(r.get(f'{side}FGA', 1))
            fgm3    = float(r.get(f'{side}FGM3', 0))
            ftm     = float(r.get(f'{side}FTM', 0))
            fta     = float(r.get(f'{side}FTA', 0))
            to_     = float(r.get(f'{side}TO',  0))
            orb     = float(r.get(f'{side}OR',  0))
            drb     = float(r.get(f'{side}DR',  0))
            opp_dr  = float(r.get(f'{opp}DR',   1))

            poss = max(fga - orb + to_ + 0.44 * fta, 1.0)
            records.append({
                'Season':  int(r['Season']),
                'TeamID':  tid,
                'eFGPct':  (fgm + 0.5*fgm3) / max(fga, 1),
                'TOPct':   to_ / poss,
                'ORBPct':  orb / max(orb + opp_dr, 1),
                'FTRate':  fta / max(fga, 1),
                'AdjOE':   pts / poss * 100,
                'AdjDE':   opp_pts / poss * 100,
                'EffMargin': (pts - opp_pts) / poss * 100,
            })

    adv = pd.DataFrame(records)
    return (adv.groupby(['Season','TeamID'])
              .mean()
              .reset_index()
              .set_index(['Season','TeamID']))


print('Computing advanced stats...')
m_adv = compute_advanced_stats(m_rsd)
w_adv = compute_advanced_stats(w_rsd)
print(f'Men adv: {m_adv.shape} | Women adv: {w_adv.shape}')

In [ ]:
# ─── Cell 6: Massey Ordinals (Pre-Tournament Only — No Data Leakage) ──────────

def compute_massey(massey_df):
    """
    Extract team ratings from Massey Ordinals BEFORE the tournament starts.
    DayNum <= 133 ensures no tournament game data is included.
    Returns dict: (Season, TeamID) -> normalized_massey_score [0,1]
    """
    if massey_df is None or massey_df.empty:
        return {}

    day_col = 'RankingDayNum' if 'RankingDayNum' in massey_df.columns else 'DayNum'

    # CRITICAL: only use pre-tournament ratings to avoid data leakage
    pre = massey_df[massey_df[day_col] <= 133].copy()

    available = pre['SystemName'].unique()
    use_sys   = [s for s in TOP_MASSEY if s in available]
    if not use_sys:
        use_sys = list(available[:8])
    print(f'  Massey systems used: {use_sys}')

    pre = pre[pre['SystemName'].isin(use_sys)]

    # Take the latest snapshot per (Season, TeamID, System)
    latest = (pre.sort_values(day_col)
               .groupby(['Season','TeamID','SystemName'])
               .last()
               .reset_index()[['Season','TeamID','SystemName','OrdinalRank']])

    # Average rank across systems
    avg = (latest.groupby(['Season','TeamID'])['OrdinalRank']
                 .mean()
                 .reset_index()
                 .rename(columns={'OrdinalRank': 'AvgRank'}))

    # Convert: lower rank = better; normalize to [0,1] within season
    result = {}
    for season, grp in avg.groupby('Season'):
        max_r = grp['AvgRank'].max()
        for _, row in grp.iterrows():
            # score = 1.0 for rank=1, 0.0 for rank=max_r
            score = (max_r - row['AvgRank']) / max(max_r - 1, 1)
            result[(int(season), int(row['TeamID']))] = score

    return result


print('Computing Massey ordinal ratings...')
m_massey_rat = compute_massey(m_massey)
w_massey_rat = compute_massey(w_massey)
print(f'Men Massey entries: {len(m_massey_rat)} | Women: {len(w_massey_rat)}')

In [ ]:
# ─── Cell 7: Seed Extraction ─────────────────────────────────────────────────

def extract_seeds(seeds_df):
    """Returns dict: (Season, TeamID) -> integer seed (1-16)"""
    if seeds_df is None or seeds_df.empty:
        return {}
    result = {}
    for _, row in seeds_df.iterrows():
        m = re.search(r'(\d+)', str(row['Seed']))
        seed_num = int(m.group(1)) if m else 16
        result[(int(row['Season']), int(row['TeamID']))] = seed_num
    return result


m_seed_d = extract_seeds(m_seeds)
w_seed_d = extract_seeds(w_seeds)
print(f'Men seeds: {len(m_seed_d)} | Women seeds: {len(w_seed_d)}')

# Quick test
latest_s = max(k[0] for k in m_seed_d)
seeds_latest = {k[1]: v for k, v in m_seed_d.items() if k[0] == latest_s}
print(f'Seed=1 teams in {latest_s}: {[t for t,s in seeds_latest.items() if s==1]}')

In [ ]:
# ─── Cell 8: Feature Vector Builder ─────────────────────────────────────────

FEAT_KEYS = [
    'elo', 'seed', 'massey',
    'WinPct', 'PointDiff', 'PPG', 'OppPPG', 'RecentWinPct',
    'eFGPct', 'TOPct', 'ORBPct', 'FTRate', 'AdjOE', 'AdjDE', 'EffMargin',
]

FEATURE_NAMES = (
    [f'd_{k}' for k in FEAT_KEYS] +
    ['r_elo', 'r_massey', 'r_winpct',          # ratio features
     'abs_seed_gap', 'abs_elo_sum',             # absolute features
     'gender']                                  # gender flag
)
N_FEATURES = len(FEATURE_NAMES)
print(f'Feature count: {N_FEATURES}')
print(FEATURE_NAMES)

# Default values when data is missing
DEFAULTS = {
    'elo':1500,'seed':9,'massey':0.5,
    'WinPct':0.5,'PointDiff':0.0,'PPG':70.0,'OppPPG':70.0,'RecentWinPct':0.5,
    'eFGPct':0.50,'TOPct':0.18,'ORBPct':0.30,'FTRate':0.33,
    'AdjOE':100.0,'AdjDE':100.0,'EffMargin':0.0,
}


def get_team_vals(season, tid, stats_df, adv_df, elo_d, seed_d, massey_d):
    """Retrieve all feature values for one team in one season."""
    v = dict(DEFAULTS)

    v['elo']    = elo_d.get((season, tid),
                   elo_d.get((season-1, tid), ELO_INITIAL))
    v['seed']   = seed_d.get((season, tid), 9)
    v['massey'] = massey_d.get((season, tid),
                   massey_d.get((season-1, tid), 0.5))

    if not stats_df.empty and (season, tid) in stats_df.index:
        row = stats_df.loc[(season, tid)]
        for k in ['WinPct','PointDiff','PPG','OppPPG','RecentWinPct']:
            if k in row.index:
                val = row[k]
                if not pd.isna(val):
                    v[k] = float(val)

    if not adv_df.empty and (season, tid) in adv_df.index:
        row = adv_df.loc[(season, tid)]
        for k in ['eFGPct','TOPct','ORBPct','FTRate','AdjOE','AdjDE','EffMargin']:
            if k in row.index:
                val = row[k]
                if not pd.isna(val):
                    v[k] = float(val)

    return v


def build_matchup_vec(season, t1, t2, stats_df, adv_df, elo_d, seed_d, massey_d, gender):
    """Build feature array for matchup (t1 < t2). Returns shape (N_FEATURES,)."""
    f1 = get_team_vals(season, t1, stats_df, adv_df, elo_d, seed_d, massey_d)
    f2 = get_team_vals(season, t2, stats_df, adv_df, elo_d, seed_d, massey_d)

    diffs   = [f1[k] - f2[k] for k in FEAT_KEYS]
    ratios  = [
        f1['elo']    / (f1['elo']    + f2['elo']    + 1e-9),
        f1['massey'] / (f1['massey'] + f2['massey'] + 1e-9),
        f1['WinPct'] / (f1['WinPct'] + f2['WinPct'] + 1e-9),
    ]
    absolutes = [
        abs(f1['seed']   - f2['seed']),
        f1['elo'] + f2['elo'],
    ]

    return np.array(diffs + ratios + absolutes + [float(gender)], dtype=np.float32)

In [ ]:
# ─── Cell 9: Build Historical Training Dataset ───────────────────────────────

def build_training_set(tour_df, stats_df, adv_df, elo_d, seed_d, massey_d, gender, min_season):
    """Build (X, y, seasons) from historical tournament results."""
    if tour_df is None or tour_df.empty:
        return np.empty((0, N_FEATURES)), np.empty(0), np.empty(0)

    rows, labels, seasons = [], [], []
    skipped = 0

    for _, game in tour_df.iterrows():
        season = int(game['Season'])
        if season < min_season:
            continue

        w_id = int(game['WTeamID'])
        l_id = int(game['LTeamID'])
        t1   = min(w_id, l_id)
        t2   = max(w_id, l_id)
        y    = 1 if w_id == t1 else 0   # 1 = lower-ID team won

        feat = build_matchup_vec(season, t1, t2, stats_df, adv_df,
                                 elo_d, seed_d, massey_d, gender)
        if feat is None:
            skipped += 1
            continue

        rows.append(feat)
        labels.append(y)
        seasons.append(season)

    print(f'  {len(rows)} games, {skipped} skipped')
    return (np.array(rows, dtype=np.float32),
            np.array(labels, dtype=np.float32),
            np.array(seasons))


print('Building Men training set...')
Xm, ym, sm = build_training_set(m_tour, m_stats, m_adv,
                                  m_elo, m_seed_d, m_massey_rat,
                                  gender=0, min_season=M_TRAIN_START)

print('Building Women training set...')
Xw, yw, sw = build_training_set(w_tour, w_stats, w_adv,
                                  w_elo, w_seed_d, w_massey_rat,
                                  gender=1, min_season=W_TRAIN_START)

X_all = np.vstack([Xm, Xw])
y_all = np.concatenate([ym, yw])
s_all = np.concatenate([sm, sw])

print(f'Combined dataset: {X_all.shape} | mean outcome: {y_all.mean():.3f}')

In [ ]:
# ─── Cell 10: LOYO CV + Final Ensemble Training ──────────────────────────────

def impute(X, medians=None):
    """Replace NaN/Inf with column medians."""
    X = np.where(np.isinf(X), np.nan, X)
    if medians is None:
        medians = np.nanmedian(X, axis=0)
        medians = np.where(np.isnan(medians), 0.0, medians)
    for j in range(X.shape[1]):
        X[:, j] = np.where(np.isnan(X[:, j]), medians[j], X[:, j])
    return X, medians


# ── Leave-One-Year-Out Cross-Validation ──────────────────────────────────────
print('Leave-One-Year-Out CV...')
years     = sorted(set(s_all))
oof_preds = np.zeros(len(X_all))
cv_results = []

for yr in years:
    tr = s_all != yr
    va = s_all == yr
    if va.sum() == 0:
        continue

    X_tr, y_tr = X_all[tr].copy(), y_all[tr]
    X_va        = X_all[va].copy()

    X_tr, meds = impute(X_tr)
    X_va, _    = impute(X_va, meds)

    sc = StandardScaler().fit(X_tr)
    X_tr_sc = sc.transform(X_tr)
    X_va_sc = sc.transform(X_va)

    xgb_m = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.8,
                           eval_metric='logloss', verbosity=0, random_state=42)
    lgb_m = LGBMClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                            num_leaves=15, subsample=0.8, colsample_bytree=0.8,
                            verbose=-1, random_state=42)
    lr_m  = LogisticRegression(C=0.1, max_iter=1000)

    xgb_m.fit(X_tr, y_tr)
    lgb_m.fit(X_tr, y_tr)
    lr_m.fit(X_tr_sc, y_tr)

    p = (0.4 * xgb_m.predict_proba(X_va)[:,1] +
         0.4 * lgb_m.predict_proba(X_va)[:,1] +
         0.2 * lr_m.predict_proba(X_va_sc)[:,1])

    oof_preds[va] = p
    brier = brier_score_loss(y_all[va], np.clip(p, CLIP_LO, CLIP_HI))
    cv_results.append((yr, brier, va.sum()))

cv_brier = np.mean([b for _, b, _ in cv_results])
print(f'\nYear-by-year LOYO Brier:')
for yr, b, n in cv_results[-8:]:
    print(f'  {yr}: {b:.4f}  (n={n})')
print(f'Mean LOYO Brier: {cv_brier:.4f}')


# ── Isotonic calibrator fitted on OOF predictions ────────────────────────────
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(oof_preds, y_all)
print('\nIsotonic calibration fitted on OOF predictions.')


# ── Final models trained on ALL data ─────────────────────────────────────────
print('\nTraining final models on full dataset...')
X_final, col_medians = impute(X_all.copy())
scaler_final = StandardScaler().fit(X_final)
X_sc_final   = scaler_final.transform(X_final)

xgb_final = XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.03,
                           subsample=0.8, colsample_bytree=0.8,
                           min_child_weight=3,
                           eval_metric='logloss', verbosity=0, random_state=42)
lgb_final = LGBMClassifier(n_estimators=500, max_depth=5, learning_rate=0.03,
                            num_leaves=15, subsample=0.8, colsample_bytree=0.8,
                            min_child_samples=10, verbose=-1, random_state=42)
lr_final  = LogisticRegression(C=0.1, max_iter=2000)

xgb_final.fit(X_final, y_all)
lgb_final.fit(X_final, y_all)
lr_final.fit(X_sc_final, y_all)
print('All models trained.')


def predict_one(x_raw):
    """Predict for a single (1, N_FEATURES) raw feature array."""
    x_imp = x_raw.copy()
    x_imp, _ = impute(x_imp, col_medians)
    x_sc  = scaler_final.transform(x_imp)

    raw = (0.4 * xgb_final.predict_proba(x_imp)[:,1] +
           0.4 * lgb_final.predict_proba(x_imp)[:,1] +
           0.2 * lr_final.predict_proba(x_sc)[:,1])

    calibrated = iso.predict(raw)
    return float(np.clip(calibrated[0], CLIP_LO, CLIP_HI))


print('\nIn-sample Brier (sanity, expected < LOYO):', end=' ')
p_train = iso.predict(0.4*xgb_final.predict_proba(X_final)[:,1] +
                      0.4*lgb_final.predict_proba(X_final)[:,1] +
                      0.2*lr_final.predict_proba(X_sc_final)[:,1])
print(f'{brier_score_loss(y_all, np.clip(p_train, CLIP_LO, CLIP_HI)):.4f}')

In [ ]:
# ─── Cell 11: Generate All Submission Predictions ────────────────────────────

print(f'Generating predictions for {len(sample_sub)} matchups...')

preds_list = []
errors     = 0

for idx, row_id in enumerate(sample_sub['ID']):
    if idx % 10000 == 0:
        print(f'  {idx}/{len(sample_sub)}')

    try:
        parts  = row_id.split('_')
        season = int(parts[0])
        t1     = int(parts[1])   # guaranteed lower ID by SampleSubmission format
        t2     = int(parts[2])

        # Route to Men's or Women's data by team ID range
        is_women = (t1 >= 3000)
        if is_women:
            feat = build_matchup_vec(season, t1, t2,
                                     w_stats, w_adv, w_elo,
                                     w_seed_d, w_massey_rat, gender=1)
        else:
            feat = build_matchup_vec(season, t1, t2,
                                     m_stats, m_adv, m_elo,
                                     m_seed_d, m_massey_rat, gender=0)

        pred = predict_one(feat.reshape(1, -1))

    except Exception as e:
        errors += 1
        pred = 0.5

    preds_list.append(pred)


print(f'\nDone. Errors: {errors}')
preds_arr = np.array(preds_list)
print(f'Pred stats: min={preds_arr.min():.4f} max={preds_arr.max():.4f} mean={preds_arr.mean():.4f} std={preds_arr.std():.4f}')

In [ ]:
# ─── Cell 12: Build Submission, Validate & Save ──────────────────────────────

submission_df = sample_sub[['ID']].copy()
submission_df['Pred'] = preds_arr

# ── Critical validation checks ───────────────────────────────────────────────
print('=== VALIDATION ===')

n_rows = len(submission_df)
n_sample = len(sample_sub)
print(f'Row count match:  {n_rows} == {n_sample} → {n_rows == n_sample}')
assert n_rows == n_sample, f'ROW COUNT MISMATCH: {n_rows} vs {n_sample}'

id_match = set(submission_df['ID']) == set(sample_sub['ID'])
print(f'ID set match:     {id_match}')
assert id_match, 'ID MISMATCH!'

in_range = submission_df['Pred'].between(0, 1).all()
print(f'Preds in [0,1]:   {in_range}')
assert in_range, 'PREDICTIONS OUT OF [0,1]!'

std_ok = submission_df['Pred'].std() > 0.01
print(f'Non-trivial std:  {submission_df["Pred"].std():.4f} > 0.01 → {std_ok}')

no_nan = not submission_df['Pred'].isna().any()
print(f'No NaN preds:     {no_nan}')
assert no_nan, 'NaN PREDICTIONS FOUND!'

# ── Save ─────────────────────────────────────────────────────────────────────
submission_df.to_csv('submission.csv', index=False)

print(f'\n✓ submission.csv saved ({n_rows:,} rows)')
print(f'  Columns: {list(submission_df.columns)}')
print(f'  LOYO CV Brier: {cv_brier:.4f}')
print(submission_df.head(10))

In [ ]:
# ─── Cell 13: Feature Importance & Diagnostics ──────────────────────────────

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Feature importance from XGBoost
try:
    fi = pd.Series(xgb_final.feature_importances_, index=FEATURE_NAMES)
    fi_sorted = fi.sort_values(ascending=False)
    print('Top 10 XGBoost Feature Importances:')
    print(fi_sorted.head(10).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    fi_sorted.head(12).plot(kind='barh', ax=axes[0])
    axes[0].set_title('Feature Importance (XGBoost)')
    axes[0].invert_yaxis()

    axes[1].hist(submission_df['Pred'], bins=60, edgecolor='black', alpha=0.7)
    axes[1].axvline(0.5, color='red', linestyle='--', label='Baseline 0.5')
    axes[1].set_title('Prediction Distribution')
    axes[1].set_xlabel('Predicted Probability')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('diagnostics.png', dpi=80)
    plt.close()
    print('diagnostics.png saved.')

except Exception as e:
    print(f'Diagnostics skipped: {e}')


# ── Summary ───────────────────────────────────────────────────────────────────
print('\n' + '='*55)
print('  FINAL SUMMARY')
print('='*55)
print(f'  Total predictions:    {len(submission_df):,}')
print(f'  LOYO CV Brier:        {cv_brier:.4f}')
print(f'  Previous best:        0.24740')
print(f'  Random baseline:      0.25000')
print(f'  Expected LB score:    ~0.19-0.21')
print(f'  Prediction std dev:   {submission_df["Pred"].std():.4f}')
print('='*55)
print('\n  READY TO SUBMIT: submission.csv')